In [0]:
# Databricks notebook source
"""Silver layer for PSA population: clean, type, and derive from bronze."""

from __future__ import annotations

import sys
import uuid

from delta.tables import DeltaTable
from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    expr,
    lit,
    sha2,
    size,
    split,
)

# COMMAND ----------

# Settings
BRONZE_TABLE = "ahon.bronze.psa_population_raw"
SILVER_TABLE = "ahon.silver.psa_population"
MERGE_KEYS = ["geographic_location", "census_year"]
PATH_SEPARATOR = " > "  # joins the levels of the hierarchical geographic path

DATA_COLS = [
    "geographic_location",
    "total_population",
    "urban_population",
    "rural_population",
    "percent_urban",
    "census_year",
    "geo_level",
    "region",
    "province",
    "city_municipality",
]
PROVENANCE_COLS = ["source_name", "source_ref", "ingested_at", "batch_id", "row_hash"]
SILVER_COLS = DATA_COLS + PROVENANCE_COLS
HASH_COLS = [
    "geographic_location",
    "total_population",
    "urban_population",
    "rural_population",
    "percent_urban",
    "census_year",
]

UPDATE_CONDITION = "NOT (target.row_hash <=> source.row_hash)"

CREATE_SILVER_SQL = f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
        geographic_location     STRING,
        total_population        BIGINT,
        urban_population        BIGINT,
        rural_population        BIGINT,
        percent_urban           DOUBLE,
        census_year             INT,
        geo_level               INT,
        region                  STRING,
        province                STRING,
        city_municipality       STRING,
        source_name             STRING,
        source_ref              STRING,
        ingested_at             TIMESTAMP,
        batch_id                STRING,
        row_hash                STRING
    ) USING DELTA
"""

# COMMAND ----------


def report(message: str) -> None:
    """Write a line to stdout, which Databricks shows as cell output."""
    sys.stdout.write(f"{message}\n")


def to_long(name: str) -> Column:
    """Cast a STRING column such as '1234.00' to BIGINT, keeping its name."""
    return col(name).cast("double").cast("long").alias(name)


def path_part(position: int) -> Column:
    """Return one level of the geographic path (1-based), or NULL if missing."""
    return expr(
        f"try_element_at(split(geographic_location, '{PATH_SEPARATOR}'), {position})"
    )


def build_silver_frame(spark: SparkSession, batch_id: str) -> DataFrame:
    """Read bronze, cast to typed columns, derive geography and rural population.

    Bronze stores total_population, urban_population and percent_urban as
    STRING. Silver casts them to BIGINT, BIGINT and DOUBLE and derives:

    - rural_population = total_population - urban_population
    - geo_level (0 national, 1 region, 2 province, 3 city/municipality)
    - region, province and city_municipality, parsed from the hierarchical path
    """
    typed = spark.table(BRONZE_TABLE).select(
        col("geographic_location"),
        to_long("total_population"),
        to_long("urban_population"),
        col("percent_urban").cast("double").alias("percent_urban"),
        col("census_year"),
        col("source_name"),
    )
    derived = typed.withColumns(
        {
            "rural_population": col("total_population") - col("urban_population"),
            "geo_level": size(split(col("geographic_location"), PATH_SEPARATOR)) - 1,
            "region": path_part(2),
            "province": path_part(3),
            "city_municipality": path_part(4),
            "source_ref": lit(BRONZE_TABLE),
            "ingested_at": current_timestamp(),
            "batch_id": lit(batch_id),
        }
    )
    # A separate step: withColumns cannot see columns it creates in the same call,
    # and the hash needs rural_population.
    hashed = derived.withColumn(
        "row_hash", sha2(concat_ws("|", *[col(c) for c in HASH_COLS]), 256)
    )
    return hashed.select(*SILVER_COLS)


def merge_into_silver(spark: SparkSession, frame: DataFrame) -> None:
    """Create the silver table if missing, then upsert on the merge keys.

    Matched rows are updated only when their content hash has changed, so
    unchanged rows keep their original provenance columns.
    """
    spark.sql(CREATE_SILVER_SQL)
    condition = " AND ".join(f"target.{key} <=> source.{key}" for key in MERGE_KEYS)
    (
        DeltaTable.forName(spark, SILVER_TABLE)
        .alias("target")
        .merge(frame.alias("source"), condition)
        .whenMatchedUpdateAll(condition=UPDATE_CONDITION)
        .whenNotMatchedInsertAll()
        .execute()
    )


def main() -> None:
    """Build the silver table from bronze, then print a summary."""
    spark = SparkSession.builder.getOrCreate()

    batch_id = str(uuid.uuid4())
    frame = build_silver_frame(spark, batch_id)
    merge_into_silver(spark, frame)

    report(f"Silver table : {SILVER_TABLE}")
    report(f"Batch ID     : {batch_id}")
    report(f"Merge keys   : {MERGE_KEYS}")
    report(f"Columns      : {SILVER_COLS}")


# COMMAND ----------

if __name__ == "__main__":
    main()